In [1]:
import pandas as pd

from core.parser.path_resolver import PathResolver
from core.data.defines_data import DefinesData
from core.data.location_data import LocationData
from core.data.location_ranks_data import LocationRanksData
from core.data.static_modifiers_data import StaticModifiersData
from core.data.pop_data import PopData
from core.data.country_setup_data import CountrySetupData
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config['game_path'], config['mod_path'])

# Base desired_pop_scaled from static_modifiers/location.txt (location_base_values)
static_modifiers_data = StaticModifiersData(path_resolver)
static_modifiers_data.load_all()
defines_data = DefinesData(path_resolver)
location_data = LocationData(path_resolver)
location_ranks_data = LocationRanksData(path_resolver)
pop_data = PopData(path_resolver)

defines_data.load_all()
location_data.load_all()
location_ranks_data.load_all()
pop_data.load_all()
pop_data_food_consumption = pop_data.modded_df[["pop_food_consumption"]]
location_df = location_data.get_merged_df()
country_setup_data = CountrySetupData(path_resolver)
country_setup_data.load_all()
country_societal_values = country_setup_data.get_societal_values_df()
desired_pop_df = location_ranks_data.get_desired_pop_df()

# From pp_location_adjustments.txt (mod) or vanilla; vanilla default 0.8
SUBSISTENCE_AGRICULTURE = defines_data.get_define("NLocation", "SUBSISTENCE_AGRICULTURE", 1.0)


# Max bonuses at -100 societal value (these stay hardcoded)
spiritualist_max = 0.02  # 2% clergy for towns/cities at spiritualist_vs_humanist = -100
aristocracy_max = 0.01  # 1% nobles for towns/cities at aristocracy_vs_plutocracy = -100

def get_actual_desired_pop(spiritualist_vs_humanist, aristocracy_vs_plutocracy):
    """actual_desired_pop from base + clergy/nobles bonuses scaled by country societal values."""
    sv = float(spiritualist_vs_humanist) if pd.notna(spiritualist_vs_humanist) else 0
    av = float(aristocracy_vs_plutocracy) if pd.notna(aristocracy_vs_plutocracy) else 0
    # -100 = full bonus, 100 = no bonus. factor = (100 - value) / 200, clamped [0,1]
    clergy_added = max(0, min(1, (100 - sv) / 200)) * spiritualist_max
    nobles_added = max(0, min(1, (100 - av) / 200)) * aristocracy_max
    result = desired_pop_df.copy()
    for col in ["town", "city"]:
        result.loc["clergy", col] += clergy_added
        result.loc["nobles", col] += nobles_added
        others_sum = result.loc[result.index != "peasants", col].sum()
        result.loc["peasants", col] = max(0.0, 1.0 - others_sum)
    return result

# Default for display (Stockholm/SWE); recomputed in location cell when analyzing a specific location
# actual_desired_pop = get_actual_desired_pop(-50, -30)

print("Base desired_pop_scaled (location_base_values):")
print(f"  local_clergy_desired_pop_scaled  = {static_modifiers_data.get_base_scaled('clergy')}")
print(f"  local_nobles_desired_pop_scaled  = {static_modifiers_data.get_base_scaled('nobles')}")

Base desired_pop_scaled (location_base_values):
  local_clergy_desired_pop_scaled  = 0.005
  local_nobles_desired_pop_scaled  = 0.0025


In [2]:
# scary modifier are global_nobles_city_desired_pop_scaled and local_nobles_desired_pop_scaled


# other things that play into is
# C:\Games\steamapps\common\Europa Universalis V\game\main_menu\common\static_modifiers\location.txt
# This is base value for every location independent of location rank
# local_clergy_desired_pop_scaled = 0.005
# local_nobles_desired_pop_scaled = 0.0025
# Same File but province capital (each province highest DEVELOPMENT location is the province capital, if there are multiple countries in a province, there must be multiple capitals, one for each country, highest dev per location per country in prvovince)
# local_nobles_desired_pop = 0.01
# local_clergy_desired_pop = 0.01
# Same for capital (chosen ONE location per country, data must be somewhere, stacks with province capital aswell)
# local_nobles_desired_pop = 0.02
# local_clergy_desired_pop = 0.02


In [3]:
# pop_data_food_consumption
# Base % of each pop type per location rank (from location_ranks, peasants = remainder)
# Use for population 1: e.g. town with pop 1 -> 25% burghers, 25% laborers, 34.5% peasants, etc.
# desired_pop_df
# Actual desired pop: base + clergy/nobles bonuses (towns/cities only)
# actual_desired_pop

In [4]:
# Country starting societal values (from setup templates + country overrides)
# location_df also has spiritualist_vs_humanist, aristocracy_vs_plutocracy per location (via owner_tag)
country_societal_values.head(15)
# specific country values
country_societal_values.loc[country_societal_values.owner_tag == "CHI"]

,owner_tag,spiritualist_vs_humanist,aristocracy_vs_plutocracy,capital
833,CHI,-25.0,-60.0,dadu


In [5]:
location_tag = "stockholm"  # e.g. "stockholm", "jiaxing"
location_row = location_data.get_location_by_tag(location_tag)
unemployed_peasants = location_row["population"]  # Proxy: total pop for now
location_rank = location_row["location_rank"]  # town, city, rural_settlement (from 07_cities_and_buildings)
town_setup = location_row.get("town_setup")  # e.g. important_scandinavian_town

# actual_desired_pop for this location's country (scaled by spiritualist/aristocracy societal values)
# Get from location_row if present, else lookup by owner_tag (handles stale modded_df)
owner = location_row.get("owner_tag")
if "spiritualist_vs_humanist" in location_row.index:
    sv, av = location_row["spiritualist_vs_humanist"], location_row["aristocracy_vs_plutocracy"]
elif pd.notna(owner):
    match = country_societal_values[country_societal_values.owner_tag == owner]
    sv = match.iloc[0]["spiritualist_vs_humanist"] if not match.empty else 0
    av = match.iloc[0]["aristocracy_vs_plutocracy"] if not match.empty else 0
else:
    sv, av = 0, 0
actual_desired_pop = get_actual_desired_pop(sv, av)

# location_df includes location_rank and town_setup
location_df[["location", "location_rank", "town_setup", "population", "development", "owner_tag"]].head(10)

,location,location_rank,town_setup,population,development,owner_tag
0,stockholm,town,important_scandinavian_town,21.816,20.0,SWE
1,norrtalje,rural_settlement,NaN,12.770,12.0,SWE
2,enkoping,rural_settlement,NaN,13.660,15.0,SWE
3,uppsala,rural_settlement,NaN,16.608,17.0,SWE
4,kastelholm,rural_settlement,NaN,5.086,20.0,SWE
5,tierp,rural_settlement,NaN,8.962,13.0,SWE
6,heby,rural_settlement,NaN,7.940,15.0,SWE
7,nykoping,rural_settlement,NaN,11.762,20.0,SWE
8,kolmarden,rural_settlement,NaN,9.008,12.0,SWE
9,strangnas,rural_settlement,NaN,9.067,13.0,SWE


In [17]:
# location_df.loc[location_df.location == "jiaxing"][["location", "location_rank", "town_setup", "area", "region", "macro_region", "super_region", "population", "development", "raw_material", "modifier", "topography", "climate", "vegetation", "religion", "culture"]]
# location_df.loc[location_df.location == "jiaxing"][["location", "owner_tag", "population", "location_rank","nobles_desired_pct", "aristocracy_vs_plutocracy", "nobles_from_aristocracy_pct", "nobles_desired_count", "clergy_desired_pct", "clergy_desired_count"]]

# location_df.loc[location_df.location == "jiaxing"][["location", "owner_tag", "population", "location_rank","nobles_desired_pct", "nobles_desired_count", "aristocracy_vs_plutocracy", "nobles_from_aristocracy_share", "nobles_from_base_location_share", "nobles_from_location_rank_share"]]
# location_df.loc[location_df.location == "jiaxing"][["location", "owner_tag", "population", "location_rank","clergy_desired_pct", "clergy_desired_count", "spiritualist_vs_humanist", "clergy_from_spiritualist_share", "clergy_from_base_location_share", "clergy_from_location_rank_share"]]
location_df.loc[location_df.location == "theodoro"][["location", "owner_tag", "population", "location_rank","clergy_desired_pct", "clergy_desired_count", "spiritualist_vs_humanist", "clergy_from_spiritualist_share", "clergy_from_base_location_share", "clergy_from_location_rank_share"]]

,location,owner_tag,population,location_rank,clergy_desired_pct,clergy_desired_count,spiritualist_vs_humanist,clergy_from_spiritualist_share,clergy_from_base_location_share,clergy_from_location_rank_share
4491,theodoro,FEO,4.846,city,0.025318,0.12269,-50.0,0.394979,0.19749,0.407531


In [7]:
location_df["food_balance_subsistence"] = location_df["food_subsistence"] - location_df["total_food_consumption"]
# Should be in percent of total consumption (avoids div-by-zero by replacing 0 with NA)
location_df["food_balance_subsistence_share"] = (
    100 * location_df["food_balance_subsistence"] / location_df["total_food_consumption"].replace(0, pd.NA)
)
location_df["food_balance_total"] = location_df["total_food_production"] - location_df["total_food_consumption"]

# Scope-level aggregates: total_consumption, total_production, from_subsistence, subsistence_pct per province/area/region/macro_region/super_region
SCOPE_COLS = ["province", "area", "region", "macro_region", "super_region"]
for scope_col in SCOPE_COLS:
    grp = location_df.groupby(scope_col, dropna=False).agg(
        total_consumption=("total_food_consumption", "sum"),
        total_production=("total_food_production", "sum"),
        from_subsistence=("food_subsistence", "sum"),
    )
    grp[f"{scope_col}_subsistence_pct"] = (
        100 * grp["from_subsistence"] / grp["total_consumption"].replace(0, pd.NA)
    )
    grp[f"{scope_col}_balance"] = grp["total_production"] - grp["total_consumption"]
    # scope_total_production / scope_total_consumption, as percent (>100 = surplus, <100 = deficit)
    grp[f"{scope_col}_balance_pct"] = (
        100 * grp["total_production"] / grp["total_consumption"].replace(0, pd.NA)
    )
    grp = grp.rename(columns={
        "total_consumption": f"{scope_col}_total_consumption",
        "total_production": f"{scope_col}_total_production",
        "from_subsistence": f"{scope_col}_from_subsistence",
    })
    location_df = location_df.merge(grp, left_on=scope_col, right_index=True, how="left")

location_df.sort_values("food_balance_total")

,super_region,macro_region,region,area,location,province,western_europe,eastern_europe,is_coastal,central_asia,...,macro_region_from_subsistence,macro_region_subsistence_pct,macro_region_balance,macro_region_balance_pct,super_region_total_consumption,super_region_total_production,super_region_from_subsistence,super_region_subsistence_pct,super_region_balance,super_region_balance_pct
8771,asia,east_asia,east_china_region,zhejiang_area,jiaxing,jiaxing_province,NaN,NaN,no,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
8614,asia,east_asia,east_china_region,jiangnan_area,huating,songjiang_province,NaN,NaN,no,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
8619,asia,east_asia,east_china_region,jiangnan_area,changshu,suzhou_province,NaN,NaN,no,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
8618,asia,east_asia,east_china_region,jiangnan_area,wuxian,suzhou_province,NaN,NaN,no,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
8775,asia,east_asia,east_china_region,zhejiang_area,dongyang,jinhua_province,NaN,NaN,no,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9870,asia,east_asia,manchuria_region,ussuri_area,yongmingcheng,suifun_province,NaN,NaN,yes,NaN,...,104104.650555,89.200794,37736.44024,132.334006,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
1849,europe,western_europe,ireland_region,leinster_area,longford,meath_province,NaN,NaN,no,NaN,...,54923.073227,89.087736,31132.530041,150.498387,86549.954671,126243.513507,77463.513507,89.501507,39693.558836,145.862022
15670,asia,south_east_asia,indonesia_region,luzon_area,pulilu,katagalugan_province,NaN,NaN,no,NaN,...,17875.34503,91.232193,11362.102275,157.989902,262438.235723,367608.273798,237708.273798,90.576845,105170.038075,140.074206
1909,europe,western_europe,ireland_region,connaught_area,cavan,breifne_province,NaN,NaN,no,NaN,...,54923.073227,89.087736,31132.530041,150.498387,86549.954671,126243.513507,77463.513507,89.501507,39693.558836,145.862022


In [20]:
location_df.loc[location_df.location == "theodoro"][["location","total_food_production", "total_food_consumption", "food_balance_total"]]

,location,total_food_production,total_food_consumption,food_balance_total
4491,theodoro,63.452273,9.736463,53.71581


In [14]:
location_df.loc[location_df.area == "greenland_area"][["location", "population"]]

,location,population
657,gardr,0.083
658,brattahlid,0.030
659,dyrnes,0.030
660,hvalsey,0.030
661,einarsfjord,0.030
662,vatnahverfi,0.030
663,herjolfsnes,0.061
664,petursvik,0.061
665,lundey,0.030
666,altafjord,0.030


In [16]:
# Per-scope summary: totals, balance, subsistence (ordered columns)
scope_col = "area"  # or "province", "area", "macro_region", "super_region"
SCOPE_SUMMARY_COLS = [
    scope_col,
    f"{scope_col}_total_consumption",
    f"{scope_col}_total_production",
    f"{scope_col}_balance",
    f"{scope_col}_balance_pct",
    f"{scope_col}_from_subsistence",
    f"{scope_col}_subsistence_pct",
]
scope_summary = (
    location_df[SCOPE_SUMMARY_COLS]
    .drop_duplicates(scope_col)
    .sort_values(f"{scope_col}_subsistence_pct", ascending=True)
)
num_cols = [c for c in SCOPE_SUMMARY_COLS if c != scope_col]
scope_summary.round(0).head(30).style.format({c: "{:.0f}" for c in num_cols}, na_rep="-").hide(axis="index")

# Example: KOR's areas with scope-level totals (each row = one area in Korea)
# location_df.loc[location_df.owner_tag == "KOR"][["area", "area_total_consumption", "area_total_production", "area_balance", "area_balance_pct", "area_from_subsistence", "area_subsistence_pct"]].drop_duplicates("area")

area,area_total_consumption,area_total_production,area_balance,area_balance_pct,area_from_subsistence,area_subsistence_pct
crimea_area,77,176,99,228,56,72
peten_area,56,105,49,187,45,80
connaught_area,63,652,589,1037,52,82
apulia_area,450,614,164,136,374,83
sicily_area,836,1237,401,148,697,83
shiewhibak_area,32,507,475,1563,27,84
sardinia_area,167,201,33,120,141,84
marche_area,512,973,461,190,433,84
valencia_area,294,310,16,105,250,85
lazio_area,831,1248,417,150,708,85
